# Семинар: Что такое "мозги" нейрона? Функции активации

**Цель семинара:** Понять, что такое функции активации, написать свои собственные и увидеть, как они помогают нейросети решать сложные задачи.

Представьте, что вы строите что-то из конструктора, но у вас есть только прямые детали. Вы сможете построить только квадратные и угловатые штуки. А чтобы построить что-то круглое или с плавными изгибами, вам понадобятся специальные, изогнутые детали.

**Функции активации — это и есть те самые "изогнутые детали" для нейросети.** Без них нейросеть может решать только очень простые, "линейные" задачи. А с ними — учиться сложным вещам, находить хитрые закономерности в данных, например, отличать котиков от собачек.

Сегодня мы:
1.  Реализуем самые важные функции активации.
2.  Посмотрим, как они влияют на обучение простой нейронной сети.
3.  Увидим, почему одни функции лучше других.

## 1. Подготовка: Наша задача и простая нейросеть

Мы будем обучать нейросеть решать задачу **XOR ("исключающее ИЛИ")**. Это задачка-загадка:
- 0 и 0  ->  **0** (оба выключены)
- 0 и 1  ->  **1** (один включен)
- 1 и 0  ->  **1** (один включен)
- 1 и 1  ->  **0** (оба включены)

Проблема в том, что вы не можете провести **одну прямую линию**, чтобы отделить красные точки (ответ 1) от синих (ответ 0). Посмотрите сами:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Входные данные (X) и правильные ответы (y) для задачи XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

# Рисуем точки
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), s=150, cmap='coolwarm', edgecolors='k')
plt.title('Задача XOR: нельзя разделить одной линией', fontsize=14)
plt.xlabel('Вход 1')
plt.ylabel('Вход 2')
plt.grid(True)
# Попытка провести линию (она всегда будет ошибаться)
plt.plot([-0.2, 1.2], [1.2, -0.2], 'k--', label='Пример неудачной линии')
plt.legend()
plt.show()

# Для решения нужна "хитрая" нейросеть с нелинейными активациями. 
# Весь код для создания сети, обучения и рисования графиков уже написан. 
# Ваша задача — написать код всего для ДВУХ методов в каждом классе: `activate` (сама функция) и `derivative` (её производная).

ОДНАКО! В реальном мире данные почти никогда не бывают такими чистыми. Они 'зашумлены', и их гораздо больше.

Поэтому мы сгенерируем **новое облако данных**: 200 точек, которые в целом следуют правилу XOR, но с небольшим случайным разбросом. Посмотрите, как они выглядят. Теперь задача стала интереснее!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def generate_noisy_xor_data(n_samples=200, noise=0.2):
    """Генерирует зашумленные данные для задачи XOR."""
    n_per_quadrant = n_samples // 4
    
    # Создаем 4 облака точек с нормальным распределением
    q1 = np.random.randn(n_per_quadrant, 2) * noise + np.array([0, 0]) # Класс 0
    q2 = np.random.randn(n_per_quadrant, 2) * noise + np.array([1, 1]) # Класс 0
    q3 = np.random.randn(n_per_quadrant, 2) * noise + np.array([0, 1]) # Класс 1
    q4 = np.random.randn(n_per_quadrant, 2) * noise + np.array([1, 0]) # Класс 1
    
    # Собираем все в один датасет
    X = np.vstack([q1, q2, q3, q4])
    y = np.vstack([
        np.zeros((n_per_quadrant * 2, 1)), # Первые два облака - класс 0
        np.ones((n_per_quadrant * 2, 1))   # Вторые два - класс 1
    ])
    
    # Перемешиваем данные, чтобы они шли не по порядку
    shuffle_idx = np.random.permutation(X.shape[0])
    return X[shuffle_idx], y[shuffle_idx]

# Генерируем и рисуем наши новые данные
X, y = generate_noisy_xor_data()

plt.figure(figsize=(7, 6))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), s=40, cmap='coolwarm', edgecolors='k', alpha=0.8)
plt.title('Наши новые данные: "Зашумленный XOR"', fontsize=14)
plt.xlabel('Вход 1')
plt.ylabel('Вход 2')
plt.grid(True)
plt.show()

# Как и раньше, ваша задача — реализовать методы `activate` и `derivative`.

In [ ]:
# ----- Вспомогательный код (уже написан для вас) -----

def run_and_plot(X, y, activation_fn, title):
    """Главная функция: создает сеть, обучает ее и рисует результат."""
    print(f'--- Обучение с {title} ---')
    
    # --- 1. Создание нейросети ---
    np.random.seed(42)
    W1 = np.random.randn(2, 4) * 0.1
    b1 = np.zeros((1, 4))
    W2 = np.random.randn(4, 1) * 0.1
    b2 = np.zeros((1, 1))
    
    # --- 2. Обучение сети ---
    history = []
    learning_rate = 0.1
    epochs = 10000
    
    for epoch in range(epochs):
        # Прямой проход (от входа к выходу)
        z1 = np.dot(X, W1) + b1
        a1 = activation_fn.activate(z1)
        z2 = np.dot(a1, W2) + b2
        output = 1 / (1 + np.exp(-z2))
        
        # Считаем ошибку
        loss = np.mean((y - output) ** 2)
        history.append(loss)
        
        # --- Магия обратного распространения ошибки ---
        d_loss_output = 2 * (output - y) / y.shape[0]
        d_output_z2 = output * (1 - output)
        d_loss_z2 = d_loss_output * d_output_z2
        d_loss_a1 = np.dot(d_loss_z2, W2.T)
        d_a1_z1 = activation_fn.derivative(z1)
        d_loss_z1 = d_loss_a1 * d_a1_z1
        
        # Обновляем веса, делая маленький шажок в нужную сторону
        W1 -= learning_rate * np.dot(X.T, d_loss_z1)
        b1 -= learning_rate * np.sum(d_loss_z1, axis=0, keepdims=True)
        W2 -= learning_rate * np.dot(a1.T, d_loss_z2)
        b2 -= learning_rate * np.sum(d_loss_z2, axis=0, keepdims=True)

    # --- 3. Рисуем красивые графики ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Результат для функции: {title}', fontsize=16)
    
    # Разделяющая поверхность
    xx, yy = np.meshgrid(np.arange(X[:,0].min()-0.1, X[:,0].max()+0.1, 0.02), 
                         np.arange(X[:,1].min()-0.1, X[:,1].max()+0.1, 0.02))
    grid_input = np.c_[xx.ravel(), yy.ravel()]
    z1_grid = np.dot(grid_input, W1) + b1; a1_grid = activation_fn.activate(z1_grid)
    z2_grid = np.dot(a1_grid, W2) + b2;  output_grid = 1 / (1 + np.exp(-z2_grid))
    Z = (output_grid > 0.5).reshape(xx.shape)
    
    ax1.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm'); ax1.scatter(X[:, 0], X[:, 1], c=y.ravel(), s=40, edgecolor='k', cmap='coolwarm', alpha=0.8)
    ax1.set_title('Как нейросеть делит мир'); ax1.set_xlabel('Вход 1'); ax1.set_ylabel('Вход 2')
    ax2.plot(history); ax2.set_title('Как быстро училась сеть'); ax2.set_xlabel('Шаг обучения'); ax2.set_ylabel('Ошибка'); ax2.set_yscale('log')
    plt.show()


## Задание 1: Сигмоида (Sigmoid)

Классическая функция. Она превращает любое число в значение от 0 до 1. Похоже на регулятор громкости: что бы ни пришло на вход, на выходе будет что-то между 0 (тишина) и 1 (максимум).

**Формула:**
$$ f(x) = \frac{1}{1 + e^{-x}} $$ 

**Производная (нужна для обучения):**
$$ f'(x) = f(x) \cdot (1 - f(x)) $$ 


Минус: в крайних положениях (близко к 0 или 1) она почти не меняется, и нейрону становится "всё равно", он перестает учиться. Это называется **насыщение**.

In [ ]:
class Sigmoid:
    def activate(self, x):
        # TODO: Реализуйте функцию Сигмоиды
        # Подсказка: формула написана выше. e в степени -x в numpy — это np.exp(-x)
        return ...

    def derivative(self, x):
        # TODO: Реализуйте производную Сигмоиды
        # Подсказка: погуглите как вычисляется производная сигмоиды
        s = self.activate(x)
        return ...


## Задание 2: Гиперболический тангенс (Tanh)

Старший брат Сигмоиды. Очень на нее похож, но работает в диапазоне от -1 до 1. То, что он центрирован вокруг нуля, часто помогает сети учиться немного быстрее.

**Формула:**
$$ f(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} $$ 
**Производная:**
$$ f'(x) = 1 - f(x)^2 $$ 
Минус: как и Сигмоида, тоже страдает от "насыщения" на краях.

In [ ]:
class Tanh:
    def activate(self, x):
        # TODO: Реализуйте функцию Tanh
        # Подсказка: в numpy уже есть готовая функция, погуглите
        return ...

    def derivative(self, x):
        # TODO: Реализуйте производную Tanh
        # Подсказка: погуглите как вычисляется производная tanh
        fx = self.activate(x)
        return ...


## Задание 3: ReLU (Rectified Linear Unit)

Суперзвезда среди функций активации. Её девиз: "Всё, что хорошо — оставляем, всё что плохо — в ноль!".
- Если на вход приходит положительное число, она его не трогает.
- Если приходит отрицательное — превращает в 0.

**Формула:**
$$ f(x) = \max(0, x) $$ 
**Производная:**
$$ f'(x) = \begin{cases} 1, & \text{if } x > 0 \\ 0, & \text{if } x \le 0 \end{cases} $$ 
Плюс: очень быстрая и помогает избежать "затухания" обучения.

In [ ]:
class ReLU:
    def activate(self, x):
        # TODO: Реализуйте функцию ReLU
        # Подсказка: в numpy есть функция 
        # которая для каждого элемента выбирает больший из двух.
        # Погуглите =)
        return ...
        
    def derivative(self, x):
        # TODO: Реализуйте производную ReLU
        # Подсказка: подумайте сами
        return (x > 0) * 1


## Задание 4: Leaky ReLU (Протекающий ReLU)

Это улучшенная версия ReLU. Она решает проблему "мёртвых нейронов". Иногда нейрон с ReLU "умирает": он всегда выдаёт ноль и перестаёт учиться. Leaky ReLU немного "протекает" в отрицательной области, давая небольшой наклон вместо нуля. Это позволяет нейрону всегда иметь шанс "ожить".

**Формула:**
$$ f(x) = \begin{cases} x, & \text{if } x > 0 \\ 0.01x, & \text{if } x \le 0 \end{cases} $$

In [ ]:
class LeakyReLU:
    def activate(self, x):
        # TODO: Реализуйте Leaky ReLU
        # Подсказка: np.where(условие, значение_если_правда, значение_если_ложь) - ваш лучший друг!
        alpha = 0.01
        return np.where(x > 0, x, x * alpha)
        
    def derivative(self, x):
        # TODO: Реализуйте производную Leaky ReLU
        # Подсказка: производная равна 1 там, где x>0, и 0.01 в остальных случаях.
        dx = ...
        return dx


## 5. Сравнение!

Теперь самое интересное. Запустим наш код для каждой из реализованных функций и посмотрим, кто справится лучше. 

**Просто запустите ячейку ниже.** Если вы всё написали правильно, вы увидите четыре набора графиков.

In [ ]:
# Создаем объекты для каждой функции активации
activations = {
    'Sigmoid': Sigmoid(),
    'Tanh': Tanh(),
    'ReLU': ReLU(),
    'Leaky ReLU': LeakyReLU()
}

# Запускаем обучение и рисуем графики для каждой из них
for name, activation_function in activations.items():
    try:
        run_and_plot(X, y, activation_function, name)
    except (TypeError, NotImplementedError):
        print(f'\nОШИБКА: Функция {name} не реализована или работает неверно. Проверьте код в ячейке выше!\n')

## Выводы

Посмотрите на графики! Что мы видим на более сложных данных?

- **Sigmoid и Tanh**: Они по-прежнему справляются, но теперь хорошо видно их "характер". Они создают очень **плавные, округлые** границы. Из-за этого им может быть сложно отделить "островки" точек другого класса, которые попали не в свой угол из-за шума. Скорость обучения у них ниже.

- **ReLU**: Справилась отлично! График ошибки падает заметно быстрее. Разделяющая граница состоит из **прямых линий**, которые образуют сложную область. Это позволяет ReLU быть более "гибким" и точно обводить нужные группы точек.

- **Leaky ReLU**: Работает почти так же хорошо, как и ReLU. На этой задаче разница может быть незаметна, но в более глубоких и сложных сетях её 'неумирающие' нейроны дают ей преимущество.

### Главный вывод:
На более реалистичных данных преимущество современных функций (ReLU и его вариантов) становится еще очевиднее. Они позволяют нейросетям учиться **быстрее и строить более сложные разделяющие поверхности**.

**Поздравляем!** Вы научили нейросеть находить закономерности в несовершенных данных, а это уже очень близко к тому, чем занимаются специалисты по машинному обучению в реальной жизни!